In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 데이터 설명
- 200 종의 새 이미지 (200개의 폴더로 나눠져 있음)

- 총 117888 장 이미지

- 평균 한 종에 대해 59.94장 정도의 이미지가 있음

- 한 폴더에는 하나의 종만 있음

In [ ]:
cp ./drive/MyDrive/Lecture/2025/sesac/CUB200.zip ./

In [ ]:
!more /etc/issue

Ubuntu 22.04.4 LTS \n \l



In [ ]:
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
address sizes

In [ ]:
!unzip CUB200.zip

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0090_82579.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0090_82579.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0065_82895.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0065_82895.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0069_82613.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0069_82613.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0046_82246.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0046_82246.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0062_84573.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0062_84573.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0041_82183.jpg  
  inflating: __MACOSX/CUB200/train/092.Nighthawk/._Nighthawk_0041_82183.jpg  
  inflating: CUB200/train/092.Nighthawk/Nighthawk_0035_84077.jpg  
  inflating: __MACOSX/CUB20

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten,Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.resnet50 import ResNet50,preprocess_input
from tensorflow.keras.preprocessing import image
import os

train_folder='CUB200/train'
test_folder='CUB200/test'

class_reduce=0.1 # 부류 수 줄여서 데이터양 줄임
no_class=int(len(os.listdir(train_folder))*class_reduce) # 부류 개수

In [ ]:
os.listdir(train_folder)

['147.Least_Tern',
 '120.Fox_Sparrow',
 '159.Black_and_white_Warbler',
 '115.Brewer_Sparrow',
 '121.Grasshopper_Sparrow',
 '065.Slaty_backed_Gull',
 '198.Rock_Wren',
 '140.Summer_Tanager',
 '062.Herring_Gull',
 '114.Black_throated_Sparrow',
 '036.Northern_Flicker',
 '051.Horned_Grebe',
 '130.Tree_Sparrow',
 '008.Rhinoceros_Auklet',
 '187.American_Three_toed_Woodpecker',
 '195.Carolina_Wren',
 '044.Frigatebird',
 '015.Lazuli_Bunting',
 '017.Cardinal',
 '143.Caspian_Tern',
 '136.Barn_Swallow',
 '155.Warbling_Vireo',
 '168.Kentucky_Warbler',
 '111.Loggerhead_Shrike',
 '149.Brown_Thrasher',
 '119.Field_Sparrow',
 '169.Magnolia_Warbler',
 '200.Common_Yellowthroat',
 '055.Evening_Grosbeak',
 '001.Black_footed_Albatross',
 '040.Olive_sided_Flycatcher',
 '087.Mallard',
 '135.Bank_Swallow',
 '071.Long_tailed_Jaeger',
 '145.Elegant_Tern',
 '021.Eastern_Towhee',
 '141.Artic_Tern',
 '024.Red_faced_Cormorant',
 '108.White_necked_Raven',
 '057.Rose_breasted_Grosbeak',
 '194.Cactus_Wren',
 '034.Gray_

In [ ]:
no_class

20

In [ ]:
x_train,y_train=[],[]
for i,class_name in enumerate(os.listdir(train_folder)):
    if i<no_class: # 20개 이하의 폴더만 선택
        for fname in os.listdir(train_folder+'/'+class_name):
            img=image.load_img(train_folder+'/'+class_name+'/'+fname,target_size=(224,224))
            if len(img.getbands())!=3:
                print("주의: 유효하지 않은 영상 발생",class_name,fname)
                continue
            x=image.img_to_array(img)
            x=preprocess_input(x)
            x_train.append(x)
            y_train.append(i)

In [ ]:
i

200

In [ ]:
img.getbands() #이미지 밴드(band) 이름을 튜플로 반환, 여기에서는 RGB 임을 나타냄

('R', 'G', 'B')

In [ ]:
fname # 20보다 큰 값이므로 train data 에는 포함되지 않는다.

'Caspian_Tern_0018_146010.jpg'

In [ ]:
class_name

'142.Black_Tern'

In [ ]:
os.listdir(train_folder)[200]

'142.Black_Tern'

In [ ]:
os.listdir(train_folder+'/'+class_name), len(os.listdir(train_folder+'/'+class_name))

(['Black_Tern_0050_144000.jpg',
  'Black_Tern_0077_144117.jpg',
  'Black_Tern_0073_144638.jpg',
  'Black_Tern_0037_144110.jpg',
  'Black_Tern_0033_144328.jpg',
  'Black_Tern_0089_144174.jpg',
  'Black_Tern_0012_144091.jpg',
  'Black_Tern_0104_144038.jpg',
  'Black_Tern_0103_143956.jpg',
  'Black_Tern_0098_144089.jpg',
  'Black_Tern_0044_144021.jpg',
  'Black_Tern_0008_143965.jpg',
  'Black_Tern_0102_144344.jpg',
  'Black_Tern_0029_144140.jpg',
  'Black_Tern_0100_144597.jpg',
  'Black_Tern_0014_143939.jpg',
  'Black_Tern_0041_144103.jpg',
  'Black_Tern_0070_144292.jpg',
  'Black_Tern_0107_144661.jpg',
  'Black_Tern_0024_144039.jpg',
  'Black_Tern_0056_143906.jpg',
  'Black_Tern_0009_144046.jpg',
  'Black_Tern_0010_144341.jpg',
  'Black_Tern_0020_144163.jpg',
  'Black_Tern_0099_144242.jpg',
  'Black_Tern_0034_144106.jpg',
  'Black_Tern_0013_143892.jpg',
  'Black_Tern_0015_143979.jpg',
  'Black_Tern_0069_144359.jpg',
  'Black_Tern_0019_144680.jpg'],
 30)

In [ ]:
y_train[-31:]

[18,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19]

In [ ]:
x_train[-3:]

[array([[[ 91.061,  34.221, -35.68 ],
         [ 93.061,  36.221, -33.68 ],
         [ 94.061,  34.221, -34.68 ],
         ...,
         [ 85.061,  23.221, -45.68 ],
         [ 81.061,  21.221, -47.68 ],
         [ 83.061,  23.221, -44.68 ]],
 
        [[ 94.061,  35.221, -35.68 ],
         [ 94.061,  34.221, -34.68 ],
         [ 92.061,  35.221, -34.68 ],
         ...,
         [ 85.061,  23.221, -45.68 ],
         [ 81.061,  19.221, -49.68 ],
         [ 83.061,  20.221, -46.68 ]],
 
        [[ 94.061,  35.221, -35.68 ],
         [ 95.061,  36.221, -34.68 ],
         [ 93.061,  36.221, -33.68 ],
         ...,
         [ 85.061,  23.221, -45.68 ],
         [ 85.061,  21.221, -47.68 ],
         [ 84.061,  21.221, -45.68 ]],
 
        ...,
 
        [[ 90.061,  40.221, -26.68 ],
         [ 91.061,  40.221, -24.68 ],
         [ 90.061,  39.221, -25.68 ],
         ...,
         [ 79.061,  26.221, -33.68 ],
         [ 79.061,  26.221, -33.68 ],
         [ 80.061,  27.221, -32.68 ]],
 
     

###preprocess_input

입력 값으로 3채널 3D나 4D 텐서를 받는다.


scaling 없이 imagenet 의 자료에 맞게 채널을 zero-centerd 한 형식으로 변형해준다.

RGB 형식에서 BGR 형식으로 변경 해준다.

scaling : 이미지에서 스케일링은 1~255 를 0 ~ 1 사이의 값으로 고쳐 주는 것을 의미한다.

zero-centerd : 0을 주변으로 음수, 양수 모두 있을 수 있다는 표현

In [ ]:
x_test,y_test=[],[]
for i,class_name in enumerate(os.listdir(test_folder)):
    if i<no_class: # 13~14행이 지정한 부류만 사용
        for fname in os.listdir(test_folder+'/'+class_name):
            img=image.load_img(test_folder+'/'+class_name+'/'+fname,target_size=(224,224))
            if len(img.getbands())!=3:
                print("주의: 유효하지 않은 영상 발생",class_name,fname)
                continue
            x=image.img_to_array(img)
            x=preprocess_input(x)
            x_test.append(x)
            y_test.append(i)

In [ ]:
x_train=np.asarray(x_train)
y_train=np.asarray(y_train)
x_test=np.asarray(x_test)
y_test=np.asarray(y_test)
y_train=tf.keras.utils.to_categorical(y_train,no_class)
y_test=tf.keras.utils.to_categorical(y_test,no_class)

#base_model.trainable=False (컨볼루션층의 가중치는 동결하여 수정이 안되게 하고, 완정연결층만 수정하게 할때 사용)

In [ ]:
base_model=ResNet50(weights='imagenet',include_top=False,input_shape=(224,224,3))

cnn=Sequential()
cnn.add(base_model)
cnn.add(Flatten())
cnn.add(Dense(1024,activation='relu'))
cnn.add(Dense(no_class,activation='softmax'))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
cnn.compile(loss='categorical_crossentropy',optimizer=Adam(0.00002),metrics=['accuracy'])
hist=cnn.fit(x_train,y_train,batch_size=16,epochs=10,validation_data=(x_test,y_test),verbose=1)

res=cnn.evaluate(x_test,y_test,verbose=0)
print("정확률은",res[1]*100)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 756s 19s/step - accuracy: 0.2689 - loss: 3.3824 - val_accuracy: 0.6263 - val_loss: 1.3251
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 700s 18s/step - accuracy: 0.9904 - loss: 0.0380 - val_accuracy: 0.7171 - val_loss: 1.1102
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 699s 18s/step - accuracy: 0.9986 - loss: 0.0040 - val_accuracy: 0.7011 - val_loss: 1.1433
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 699s 18s/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.7028 - val_loss: 1.1722
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 676s 18s/step - accuracy: 1.0000 - loss: 7.5364e-04 - val_accuracy: 0.7011 - val_loss: 1.1879
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 664s 17s/step - accuracy: 1.0000 - loss: 6.2423e-04 - val_accuracy: 0.7064 - val_loss: 1.1965
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 709s 19s/step - accuracy: 1.0000 - loss: 8.3498e-04 - val_accuracy: 0.7011 - val_loss: 1.2016
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 700s 19s/step - accuracy: 1.0000 - loss: 2.5188e-04 - 